# SPAR — 4 composites x 7 tiles, gross/net toggle, as-of 0CQ

Fabric **Python** notebook (not PySpark). Pulls SPAR Engine results for four GIPS
composite ACCTs against each one's default benchmark, across seven saved SPAR components
("tiles"), on a gross / net / both basis, and lands them in `hbcm_datahub`.

```
grid = TILES x STRATEGIES x FEE_BASIS  ->  one SPAR calculation unit each
     = 7     x 4          x 2          =  56 units, submitted as 7 per-tile batches
```

## The tiles

`enddate` is **`0CQ`** — most recent calendar quarter end — for every tile. Only
`startdate` varies.

| Tile | Start | Peer universe |
|---|---|---|
| `multi_horizon_returns` | inception | — |
| `calendar_year_returns` | inception | — |
| `cumulative_monthly` | inception | — |
| `monthly_raw_returns` | inception | — |
| `peer_multi_horizon` | inception | **yes** |
| `peer_calendar_year` | inception | **yes** |
| `risk_stats_3y` | −3Y | — |
| `risk_stats_itd` *(planned)* | inception | — |

Risk stats want a fixed 3-year window while the cumulative, calendar-year, multi-horizon
and monthly tiles want inception-to-date, so `startdate` is per tile.
`startdate: "INCEPTION"` resolves per-strategy from that ACCT's inception date, which is
how the planned since-inception risk variant reuses the same component id under a second
tile key.

`universeid` is sent **only** for the two peer tiles. On the others it is omitted from
the payload entirely rather than sent as null, which would override the component's
saved universe with nothing.

## Dates

`0CQ` uses FactSet's calendar-period grammar (`CQ` / `CY` for calendar, `FQ` / `FY` for
fiscal), so it's unambiguous where a bare `0Q` is not. SPAR Engine has no `DatesApi`, so
relative tokens resolve server-side only and can't be verified before submitting.

`AS_OF_ABS` — the most recent completed calendar quarter end, computed in plain Python —
always labels the output, so landed rows carry a real date rather than a token. Cell 3
prints both; if they ever disagree the labels are wrong. Set
`USE_ABSOLUTE_AS_OF = True` to send the absolute date instead of `0CQ`, which is what you
want for a backfill or restatement re-run where the request should be identical on every
replay rather than drifting as quarters roll.

## Fee basis

Each ACCT carries **both** a gross and a net return stream, so the toggle is
`SPARIdentifier.returntype` — one account id, two return types. Set
`SYMBOL_SUFFIX_MODE = True` for the alternative QR/`$$Performance.ofdb` layout where
gross and net are separate `_G` / `_N` symbols.

Return type values are often numeric codes, not the literal strings `"Gross"` / `"Net"`.
Cell 4a prints what the API reports per ACCT — take the values from there.

## What you must fill in before first run

All of it is workstation-sourced, all in Cell 3, and Cell 4 discovers every piece:

| Field | Discovery |
|---|---|
| `STRATEGIES[code]["acct"]` + `["prefix"]` | Cell 4a |
| `STRATEGIES[code]["returntype"]` (gross, net) | Cell 4a |
| `STRATEGIES[code]["benchmark"]` (id, prefix) | Cell 4e — **a wrong prefix returns a bare 400 with no message** |
| `STRATEGIES[code]["inception"]` | from the composite record |
| `STRATEGIES[code]["universe"]` | Cell 4c |
| `TILES[tile]["componentid"]` | Cell 4b |
| `TILES[tile]["frequency"]` | Cell 4d |

Cell 3 asserts that every peer tile has a universe and every `INCEPTION` tile has an
inception date, so bad config fails before an API call is spent.

## Versions — latest as of 2026-08-06

| Package | Version | Notes |
|---|---|---|
| `fds.sdk.SPAREngine` | **3.0.0** | released 2026-05-20 |
| `fds.sdk.utils` | 3.0.1 | OAuth helper |
| `fds.protobuf.stach.extensions` | 1.3.3 | STACH parsing |
| `deltalake` (delta-rs) | preinstalled | OneLake write — do NOT pin or reinstall; the Python notebook runtime ships delta-rs and duckdb, and pinning risks a downgrade |

SPAREngine 3.0.0 is a **major bump for runtime hygiene, not an API redesign**. Per
FactSet's `BREAKING.md` (2026-05-20) every Python SDK was bumped together: Python
3.7/3.8/3.9 support dropped, and `urllib3` moved from `>=1.25.3,<2.1.0` to `>=2.7.0`.
Every model and method this notebook touches is unchanged between the 2.x docs and 3.0.0.

Minimum Python is now **3.10**. Fabric Python notebook kernels are 3.10 / 3.11 / 3.12
with **3.12 the default**, so any current kernel satisfies it — prefer 3.12, since 3.10
reaches end of support in October 2026.

> ⚠️ The SPAR SDK vendored under `code/python/SPAREngine/v3/` **in this repo is 2.0.3**
> (last synced 2025-07-21) and predates both the `universeid` field and
> `SPARPeerUniverseApi`. Verify any SPAR question against upstream `main`, not the local
> mirror. PA Engine is at **4.0.0** upstream and carries a second, separate breaking
> change (2026-07-21: required fields dropped from `PADateParameters`) that matters for
> the PA side of the pipeline but not for this notebook.

## Library setup — read this before scheduling

Do **not** rely on `%pip install` here. Per Microsoft, inline installs are *disabled by
default in pipeline runs* and *unsupported in reference runs*, and `%pip`-installed
libraries are not retained across runs. Attach the four packages above to a **Fabric
Environment** and bind this notebook to it.

For interactive first-run only:
```
%pip install fds.sdk.SPAREngine==3.0.0 fds.sdk.utils==3.0.1 \
             fds.protobuf.stach.extensions==1.3.3
```

In [ ]:
# === Cell 1: credentials ===================================================
# HBCM_Config defines FACTSET_USER and FACTSET_APIKEY as plain strings.
# %run works in both interactive and pipeline mode; notebookutils.notebook.run() does not
# propagate variables, so keep this as %run.
%run HBCM_Config

In [ ]:
# === Cell 2: imports + API client ==========================================
import json, time, datetime as dt
import pandas as pd

import fds.sdk.SPAREngine
from fds.sdk.SPAREngine.api import (
    spar_calculations_api,
    spar_peer_universe_api,   # SDK >= 2.1; absent from the 2.0.3 copy vendored here
    accounts_api,
    components_api,
    benchmarks_api,
    frequencies_api,
)
from fds.sdk.SPAREngine.models import (
    SPARCalculationParametersRoot,
    SPARCalculationParameters,
    SPARIdentifier,
    SPARDateParameters,
    CalculationMeta,
)
# PA Engine is imported ONLY to resolve relative dates. SPAR has no DatesApi, so PA is the
# authority on what 0CQ means, and using it keeps both pipelines on one as-of date.
import fds.sdk.PAEngine
from fds.sdk.PAEngine.api import dates_api as pa_dates_api

from deltalake import DeltaTable, write_deltalake
from urllib3 import Retry   # SDK 3.0.0 requires urllib3 >= 2.7.0

# Fail loudly on a stale Environment rather than 400-ing later on universeid.
# Compare the major as an int — a string compare would rank "10.0.0" below "3.0.0".
SDK_VERSION = fds.sdk.SPAREngine.__version__
assert int(SDK_VERSION.split(".")[0]) >= 3, (
    f"fds.sdk.SPAREngine {SDK_VERSION} found; this notebook targets >=3.0.0. "
    "Check the bound Fabric Environment."
)
print("SPAREngine SDK", SDK_VERSION)

configuration = fds.sdk.SPAREngine.Configuration(
    username=FACTSET_USER,
    password=FACTSET_APIKEY,
)
# Preferred once an app-config.json exists in Key Vault:
#   from fds.sdk.utils.authentication import ConfidentialClient
#   configuration = fds.sdk.SPAREngine.Configuration(
#       fds_oauth_client=ConfidentialClient(str(config_path)))

# Retry server-side failures only. Never add 429 with a naive backoff — urllib3 already
# honours Retry-After for 429, and 4xx other than 429 will not fix themselves.
configuration.retries = Retry(
    total=3,
    status_forcelist=[500, 502, 503, 504],
    backoff_factor=2,
    allowed_methods=frozenset(["GET", "POST"]),
)

api_client = fds.sdk.SPAREngine.ApiClient(configuration)
calc_api = spar_calculations_api.SPARCalculationsApi(api_client)

# Separate client for the PA date lookup — same credentials, different SDK package.
pa_configuration = fds.sdk.PAEngine.Configuration(
    username=FACTSET_USER, password=FACTSET_APIKEY,
)
pa_api_client = fds.sdk.PAEngine.ApiClient(pa_configuration)
pa_d_api = pa_dates_api.DatesApi(pa_api_client)

# --- run metadata ----------------------------------------------------------
# Populated as the notebook proceeds and landed in the final cell. Request keys are what
# FactSet support needs to retrieve the exact request; they only exist at call time.
RUN_META = {
    "run_started_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "notebook": "spar_composite_returns",
    "engine": "SPAREngine",
    "sdk_version": SDK_VERSION,
    "pa_sdk_version": fds.sdk.PAEngine.__version__,
}

INTERESTING_HEADERS = (
    "X-DataDirect-Request-Key",
    "X-FactSet-Api-Request-Key",
    "X-FactSet-Api-RateLimit-Limit",
    "X-FactSet-Api-RateLimit-Remaining",
    "X-FactSet-Api-RateLimit-Reset",
)

def call_with_headers(fn, *args, **kwargs):
    """Call fn's _with_http_info sibling and return (result, headers).

    The wrapper-returning endpoints don't document their _with_http_info shape, so handle
    both a 3-tuple and a bare return, and degrade to no headers rather than failing the
    run over telemetry.
    """
    sibling = getattr(fn.__self__, fn.__name__ + "_with_http_info", None)
    if sibling is None:
        return fn(*args, **kwargs), {}
    try:
        out = sibling(*args, **kwargs)
    except TypeError:
        return fn(*args, **kwargs), {}
    if isinstance(out, tuple) and len(out) == 3:
        result, _status, headers = out
        return result, {k: v for k, v in dict(headers or {}).items()
                        if k in INTERESTING_HEADERS}
    return out, {}

# --- STACH parsing, driven by the package's own schema ---------------------
# The API is builder-based: get_row_organized_builder -> set_package -> build ->
# convert_to_dataframe. (There is no get_stach_extension/convert pair — calling that raises
# AttributeError.) `contentorganization` is a row-organized setting here, hence the row
# builder; a Column setting needs get_column_organized_builder.
from fds.protobuf.stach.extensions.StachExtensionFactory import StachExtensionFactory
from fds.protobuf.stach.extensions.StachVersion import StachVersion
from fds.protobuf.stach.v2.RowOrganized_pb2 import RowOrganizedPackage

# STACH declares a type per column, so dates and measures are identified from the package
# rather than sniffed from values or guessed from column names.
STACH_DATE_TYPES = {"date", "datetime", "timestamp"}
STACH_NUMERIC_TYPES = {"double", "float", "int", "integer", "int32", "int64", "long",
                       "decimal", "number", "percent"}

def _row_builder():
    return StachExtensionFactory.get_row_organized_builder(StachVersion.V2)

def _as_package(payload):
    """Parse a result payload into a RowOrganizedPackage, tolerating a wrapper key.

    A wrapper would otherwise yield a package with zero tables and an empty DataFrame —
    silently, which is the worst outcome. Fail with something actionable instead.
    """
    if hasattr(payload, "to_dict"):
        payload = payload.to_dict()
    candidates = [payload]
    if isinstance(payload, dict) and isinstance(payload.get("data"), dict):
        candidates.append(payload["data"])
    for cand in candidates:
        pkg = _row_builder().set_package(cand).get_package()
        if len(pkg.tables):
            return pkg
    raise ValueError(
        "no STACH tables in the response. Confirm meta.format='JsonStach' and a "
        "row-organized contentorganization ('SimplifiedRow' or 'Row')."
    )

def stach_parse(payload):
    """-> list of (table_id, DataFrame, column_schema, group_levels).

    column_schema: {column: {stach_type, is_dimension, is_hidden, format, null_format}}
    group_levels:  per body row, the STACH hierarchy level (0 = outermost group) or None.
                   This is the authoritative grain discriminator for GROUPSALL output — no
                   guessing at a level column or a null security id.

    Each table is converted through its own single-table extension so the DataFrame is
    guaranteed to pair with the table it came from; iterating a protobuf map twice and
    zipping would depend on map ordering, which is unspecified.
    """
    pkg = _as_package(payload)
    out = []
    for tid in list(pkg.tables.keys()):
        table = pkg.tables[tid]
        df = _row_builder().add_table(tid, table).build().convert_to_dataframe()[0]
        # convert_to_dataframe names columns `description or name` — match that exactly.
        schema = {
            (c.description or c.name): {
                "stach_type": (c.type or "").lower(),
                "is_dimension": bool(c.is_dimension),
                "is_hidden": bool(c.is_hidden),
                "format": c.format.format or "",
                "null_format": c.format.null_format or "",
            }
            for c in table.definition.columns
        }
        levels = []
        for r in table.data.rows:
            if RowOrganizedPackage.Row.RowType.Name(r.row_type) == "Header":
                continue
            lv = None
            for _k, detail in r.cell_details.items():
                lv = int(detail.group_level)
                break
            levels.append(lv)
        out.append((tid, df, schema, levels))
    return out

def type_from_schema(df, schema, never_numeric=frozenset()):
    """Type a STACH frame from its declared schema. -> (df, report).

    Three things come from STACH instead of guesswork:
      - the declared `type` decides date vs numeric vs text
      - `is_dimension` protects identifiers, so an FSYM id or a zero-padded code is never
        coerced to a float
      - `null_format` is the exact token meaning "no value" for THAT column, rather than a
        global guess at "--" / "N/A"

    A column with no declared type falls back to sniffing and is reported, so an undeclared
    measure is visible rather than quietly landing as text.
    """
    out = df.copy()
    report = {"date": [], "numeric": [], "string": [], "undeclared": []}
    for c in out.columns:
        meta = schema.get(c, {})
        stype = meta.get("stach_type", "")
        txt = out[c].astype("string").str.strip()
        nullfmt = meta.get("null_format", "")
        drop = {"", "--", "N/A", "NA", "n/a", "None", "nan"} | ({nullfmt} if nullfmt else set())
        txt = txt.where(~txt.isin(drop), pd.NA)

        if c in never_numeric or meta.get("is_dimension"):
            out[c] = txt
            report["string"].append(c)
        elif stype in STACH_DATE_TYPES:
            out[c] = pd.to_datetime(txt, errors="coerce", format="mixed").dt.date
            report["date"].append(c)
        elif stype in STACH_NUMERIC_TYPES:
            out[c] = pd.to_numeric(txt.str.replace(",", "", regex=False).str.rstrip("%"),
                                   errors="coerce").astype("Float64")
            report["numeric"].append(c)
        else:
            report["undeclared"].append(f"{c}:{stype or 'no type'}")
            conv = pd.to_numeric(txt.str.replace(",", "", regex=False).str.rstrip("%"),
                                 errors="coerce")
            nn = txt.notna()
            if not nn.any():
                # All-blank: type numerically so the Delta schema is stable across quarters.
                # Landing text now and a float the quarter it populates cannot be merged.
                out[c] = pd.Series(pd.NA, index=out.index, dtype="Float64")
                report["numeric"].append(c)
            elif (conv.notna() & nn).sum() / nn.sum() >= 0.90:
                out[c] = conv.astype("Float64")
                report["numeric"].append(c)
            else:
                out[c] = txt
                report["string"].append(c)
    return out, report

def schema_date_columns(schema):
    """Columns STACH declares as dates — used instead of matching column names."""
    return [c for c, m in schema.items() if m.get("stach_type", "") in STACH_DATE_TYPES]

def dedupe_columns(df, provenance):
    """Rename any STACH column that collides with a provenance column.

    Normalising STACH labels to snake_case can land one on top of an inserted column —
    "Currency" -> currency, for instance. Duplicate names break the Delta write, and
    silently shadow one of the two before that.
    """
    clash = [c for c in df.columns if c in provenance]
    return (df.rename(columns={c: f"{c}_src" for c in clash}), clash) if clash else (df, [])

def table_exists(path):
    """True/False, distinguishing 'no such table' from a real failure.

    A bare try/except around DeltaTable() that falls back to overwrite will destroy every
    prior quarter the first time a transient auth or throttling error shows up.
    """
    try:
        DeltaTable(path)
        return True
    except Exception as e:
        msg = f"{type(e).__name__}: {e}".lower()
        if any(k in msg for k in ("not a delta table", "no log files", "not found",
                                  "does not exist", "notfound", "no such file")):
            return False
        raise RuntimeError(
            f"could not determine whether {path} exists ({e!r}). Refusing to continue: "
            f"treating this as a missing table would overwrite existing history."
        ) from e

# --- strategy_code: the one join key across every PA and SPAR table --------
# The semantic model is filtered to a single strategy at a time, so strategy_code has to be
# present, populated and identically spelled on every table. Declared canonically here and
# asserted on both sides rather than trusted.
STRATEGY_CODES = ("LC", "LCS", "SMID", "CONC")
STRATEGY_LABELS = {
    "LC":   "Large Cap",
    "LCS":  "Large Cap Select",
    "SMID": "SMID",
    "CONC": "Concentrated Equity",
}

def assert_strategy_key(df, table_name, expected=STRATEGY_CODES):
    """strategy_code must be present, fully populated, and in the canonical set.

    A null or off-spec code does not fail loudly downstream — it produces a row that
    silently disappears from every strategy-filtered visual, which is worse than an error.
    """
    assert "strategy_code" in df.columns, f"{table_name}: no strategy_code column"
    s = df["strategy_code"].astype("string")
    n_null = int(s.isna().sum())
    assert n_null == 0, f"{table_name}: {n_null} rows have no strategy_code"

    bad_fmt = sorted(set(s[~s.str.fullmatch(r"[A-Z0-9]{2,4}")].dropna()))
    assert not bad_fmt, (
        f"{table_name}: strategy_code must be 2-4 upper-case alphanumerics; got {bad_fmt}"
    )
    unknown = sorted(set(s.dropna()) - set(expected))
    assert not unknown, (
        f"{table_name}: strategy_code values not in STRATEGY_CODES: {unknown}. "
        f"Either add them to the canonical list or fix the mapping."
    )
    missing = sorted(set(expected) - set(s.dropna()))
    if missing:
        # Not fatal — a tile may legitimately not cover every strategy — but a silently
        # absent strategy looks identical to one with no data.
        print(f"  NOTE {table_name}: no rows for {missing}")
    return sorted(set(s.dropna()))

def assert_unique_grain(df, keys, table_name):
    """The stated grain must actually be unique.

    A duplicate on the declared key is what makes a Power BI relationship fan out and
    weights double — and it shows up as plausible-but-wrong numbers, not as an error.
    """
    present = [k for k in keys if k in df.columns]
    missing = [k for k in keys if k not in df.columns]
    if missing:
        print(f"  NOTE {table_name}: grain columns absent, cannot verify: {missing}")
        return None
    dup = df.duplicated(subset=present, keep=False)
    n = int(dup.sum())
    if n:
        print(f"  *** {table_name}: {n} rows duplicate the declared grain {present}")
        print(df.loc[dup, present].head(12).to_string(index=False))
    assert n == 0, (
        f"{table_name}: {n} rows share a {present} key. Landing this would make any "
        f"relationship on those columns fan out."
    )
    print(f"  OK   {table_name}: unique on {present} ({len(df)} rows)")
    return True

# --- vintage bundling ------------------------------------------------------
# asof_date (YYYYMMDD) is the VINTAGE key: every table written by either notebook in a
# quarter carries the same value, so the whole data pack is one auditable bundle. Power BI
# should read the latest COMPLETE vintage, which is what dim_vintage exposes — a vintage
# where PA landed but SPAR failed is present but incomplete, and consuming it would show
# holdings against last quarter's returns.
EXPECTED_VINTAGE_TABLES = (
    "spar_composite_returns",
    "pa_sector_weights",
    "pa_security_weights",
    "pa_characteristics",
)

def record_vintage(entries, notebook):
    """Append manifest rows, then rebuild dim_vintage from the full manifest.

    Reads asof_tag / TABLE_VINTAGE_MANIFEST / TABLE_DIM_VINTAGE from the notebook globals at
    call time — they are set in Cells 3 and 4b, which run before this is ever invoked.

    entries: {table_name: row_count}. Rebuilding from the manifest rather than tracking
    state means whichever notebook runs last produces the correct completeness, with no
    ordering assumption beyond both having run.
    """
    now = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")
    rows = [{"asof_date": asof_tag, "table_name": t, "row_count": int(n),
             "notebook": notebook, "written_utc": now} for t, n in entries.items()]
    manifest_add = pd.DataFrame(rows).astype("string")
    manifest_add["row_count"] = [int(n) for n in entries.values()]

    if table_exists(TABLE_VINTAGE_MANIFEST):
        # Replace this notebook's rows for this vintage so a re-run does not duplicate.
        DeltaTable(TABLE_VINTAGE_MANIFEST).delete(
            f"asof_date = '{asof_tag}' AND notebook = '{notebook}'")
        write_deltalake(TABLE_VINTAGE_MANIFEST, manifest_add, mode="append",
                        schema_mode="merge")
        manifest = DeltaTable(TABLE_VINTAGE_MANIFEST).to_pandas()
    else:
        write_deltalake(TABLE_VINTAGE_MANIFEST, manifest_add, mode="overwrite",
                        schema_mode="overwrite")
        manifest = manifest_add.copy()

    present = manifest.groupby("asof_date")["table_name"].agg(lambda s: sorted(set(s)))
    dim = pd.DataFrame({
        "asof_date": present.index,
        "tables_present": [",".join(v) for v in present],
        "n_tables": [len(v) for v in present],
        "is_complete": [set(EXPECTED_VINTAGE_TABLES) <= set(v) for v in present],
    }).sort_values("asof_date")
    dim["asof_date_iso"] = pd.to_datetime(dim["asof_date"], format="%Y%m%d").dt.date
    complete = dim.loc[dim["is_complete"], "asof_date"]
    latest_complete = complete.max() if len(complete) else None
    dim["is_latest_complete"] = dim["asof_date"] == latest_complete
    dim["is_latest"] = dim["asof_date"] == dim["asof_date"].max()

    write_deltalake(TABLE_DIM_VINTAGE, dim, mode="overwrite", schema_mode="overwrite")
    print(f"\nvintage {asof_tag}: {', '.join(f'{t}={n}' for t, n in entries.items())}")
    print(dim.to_string(index=False))
    if latest_complete is None:
        print("*** no vintage is complete yet — expected "
              f"{list(EXPECTED_VINTAGE_TABLES)}. Run the other notebook before pointing")
        print("*** Power BI at this data.")
    elif latest_complete != asof_tag:
        print(f"*** this vintage ({asof_tag}) is NOT yet complete; latest complete is "
              f"{latest_complete}. Power BI should stay on {latest_complete}.")
    return dim


In [ ]:
# === Cell 3: THE CONFIG BLOCK — the only cell you normally edit ============

CURRENCY = "USD"            # all GIPS series are USD-denominated

# --- as-of date (uniform across every tile) --------------------------------
# "0CQ" = most recent calendar quarter end, in FactSet's calendar-period grammar
# (CQ / CY calendar, FQ / FY fiscal) — unambiguous where a bare "0Q" is not.
#
# SPAR has no DatesApi and cannot resolve relative tokens. Cell 4b asks PA to resolve 0CQ
# and SENDS THE ABSOLUTE DATE, which makes the request reproducible on replay and puts both
# pipelines on one as-of. Only `startdate` varies per tile; `enddate` is global.
AS_OF_RELATIVE = "0CQ"

# What gets SENT. Dynamic by default: 0CQ rolls forward on its own, so the scheduled job
# needs no maintenance. The absolute date PA resolves (Cell 4b) is used to LABEL rows and to
# validate what came back — dynamic request, pinned label.
# Flip to True for a backfill or restatement replay, where the request must be identical
# every time rather than tracking the current quarter.
SEND_ABSOLUTE_END_DATE = False

# Superseded per strategy when a tile sets useeachportfolioinception (see TILES). SPAR
# requires a startdate even then, so this is a wide floor, not a real window.
INCEPTION_FALLBACK_START = "-30Y"

# --- fee basis toggle ------------------------------------------------------
# Each ACCT carries BOTH a gross and a net return stream, so the toggle is
# SPARIdentifier.returntype — one account id, two return types. (Contrast the QR/OFDB
# layout where gross and net are separate GIPS_<STRAT>_G / _N symbols; that path is
# still available via SYMBOL_SUFFIX_MODE below.)
#
# returntype values go in STRATEGIES[...]["returntype"]. They are often numeric codes
# rather than the literal strings "Gross"/"Net" — Cell 4a prints what the API reports.
# Fee basis is set PER TILE (see TILES) rather than globally, because the requirement
# differs by widget:
#   - Performance tiles carry BOTH. SEC Marketing Rule requires net alongside any gross
#     presentation, so a returns widget that can only show gross is not usable.
#   - Risk statistics and peer tables default to GROSS ONLY. Risk stats are not returns, so
#     the net requirement does not bite, and peer universes are conventionally gross —
#     ranking a net return against a gross universe is not comparable.
#
# That peer default is a COMPLIANCE JUDGEMENT, not a technical one: a peer table ranking
# returns is arguably a performance presentation. If compliance wants net alongside it,
# change that tile's "bases" to ("gross", "net") — one word, and the unit count grows by 4.
DEFAULT_BASES = ("gross", "net")
SYMBOL_SUFFIX_MODE = False  # True => append _G/_N to the account id instead
SUFFIX = {"gross": "_G", "net": "_N"}

# --- benchmark + peer universe groups --------------------------------------
# LC and LCS share the Russell 1000 benchmark AND the same peer universe, so that pair
# is defined once here and referenced by both. One source of truth, so the two can never
# drift apart in config.
#
# This is a config-level share, NOT a request-level one. See the note on `bench_group`
# in STRATEGIES below for why the two strategies still get separate calculation units.
# Three distinct benchmarks across four composites:
#   r1000  Russell 1000  -> LC, LCS   (the shared pair)
#   r2500  Russell 2500  -> SMID
#   r3000  Russell 3000  -> CONC
# Only the ids and prefixes are still TODO. Confirm each with BenchmarksApi (Cell 4e)
# before the first real run — a wrong prefix is a bare 400 with no diagnostic.
BENCHMARK_GROUPS = {
    "r1000": {   # Russell 1000
        "benchmark": {"id": "<TODO>", "prefix": "RUSSELL:", "returntype": None},
        "universe":  "<TODO shared LC/LCS peer universe>",
    },
    "r2500": {   # Russell 2500
        "benchmark": {"id": "<TODO>", "prefix": "RUSSELL:", "returntype": None},
        "universe":  "<TODO>",
    },
    "r3000": {   # Russell 3000
        "benchmark": {"id": "<TODO>", "prefix": "RUSSELL:", "returntype": None},
        "universe":  "<TODO>",
    },
}

# --- the four composites ---------------------------------------------------
# TODO — acct, prefix, returntype and inception per ACCT. Cell 4 discovers all of them.
#
# `bench_group` points at BENCHMARK_GROUPS above. LC and LCS both point at "r1000";
# SMID is Russell 2500 and CONC is Russell 3000, so LC + LCS are confirmed as the pair.
#
# Sharing a benchmark does NOT mean sharing a calculation unit. SPAR takes one `dates`
# object per unit, so two strategies in one unit must share a start date — which is
# incompatible with resolving INCEPTION per strategy. Explicit per-strategy inception is
# worth more than saving a handful of units, so each strategy keeps its own unit.
STRATEGIES = {
# No `inception` field: the engine derives each strategy's start from its own available
# monthly data via useeachportfolioinception, and the ACTUAL dates are read back off the
# response (Cell 8). Nothing to maintain, and the window is confirmed rather than asserted.
    "LC": {
        "label":       "Large Cap",
        "acct":        "<TODO>",
        "prefix":      "CLIENT:",
        "returntype":  {"gross": "<TODO>", "net": "<TODO>"},
        "bench_group": "r1000",       # Russell 1000, shared with LCS
    },
    "SMID": {
        "label":       "SMID",
        "acct":        "<TODO>",
        "prefix":      "CLIENT:",
        "returntype":  {"gross": "<TODO>", "net": "<TODO>"},
        "bench_group": "r2500",       # Russell 2500
    },
    "LCS": {
        "label":       "Large Cap Select",
        "acct":        "<TODO>",
        "prefix":      "CLIENT:",
        "returntype":  {"gross": "<TODO>", "net": "<TODO>"},
        "bench_group": "r1000",          # shares with LC
    },
    "CONC": {
        "label":       "Concentrated Equity",
        "acct":        "<TODO>",
        "prefix":      "CLIENT:",
        "returntype":  {"gross": "<TODO>", "net": "<TODO>"},
        "bench_group": "r3000",       # Russell 3000
    },
}

assert tuple(STRATEGIES) == STRATEGY_CODES, (
    f"STRATEGIES keys {tuple(STRATEGIES)} must match the canonical STRATEGY_CODES "
    f"{STRATEGY_CODES} — this is the join key for every table in the model"
)
for _c, _s in STRATEGIES.items():
    assert _s["label"] == STRATEGY_LABELS[_c], f"{_c}: label disagrees with STRATEGY_LABELS"

def strategy_config(code: str) -> dict:
    """Strategy record with its benchmark group flattened in."""
    s = dict(STRATEGIES[code])
    s.update(BENCHMARK_GROUPS[s["bench_group"]])
    return s

# --- the tiles -------------------------------------------------------------
# A "tile" is a saved SPAR component. It fixes the statistic columns server-side; the
# POST body acts as a one-time override of the component's saved dates.
#
#   startdate       "INCEPTION" sets useeachportfolioinception, so the ENGINE starts each
#                   strategy at its own earliest available monthly data — no configured
#                   dates to maintain and no strategy silently truncated to another's
#                   history. Everything that can use it does; the risk tile needs a fixed
#                   comparable window, so it sends -3Y instead.
#                   enddate is END_DATE for every tile.
#   bases           Which fee bases to run. Defaults to both; risk and peer tiles override.
#   time_series     Dated series rather than one row per period label. Cell 8 derives the
#                   real start/end/as-of per strategy from these.
#   needs_universe  True  -> universeid is sent, and the group must have one.
#                   False -> universeid omitted entirely (never sent as null).
#   frequency       Confirm against FrequenciesApi (Cell 4c).
#
# A component id may appear more than once under different variant keys — that is how
# the eventual "risk stats since inception" variant sits alongside the 3Y one.
#   component_name  The component's name AS SAVED IN THE WORKSTATION. This is the
#                   contract, not the id — Cell 4 resolves name -> id on every run,
#                   because re-saving a component can mint a new id and a stale
#                   hardcoded id 400s with no hint that the id is what broke.
#   pinned_componentid  Optional. Last known id. Purely for drift detection: if the
#                   resolved id differs, Cell 4 shouts, because a re-saved component may
#                   also have had its columns changed. Leave None on first run, then
#                   paste in what Cell 4 printed.
#   time_series     True for tiles whose output is a dated series rather than one row
#                   per period-label. Used for the ragged-series report in Cell 8.
TILES = {
    # --- performance: BOTH bases, per SEC Marketing Rule ---------------------
    "multi_horizon_returns": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "startdate": "INCEPTION", "frequency": "Monthly",
        "bases": DEFAULT_BASES, "needs_universe": False, "time_series": False,
    },
    "calendar_year_returns": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "startdate": "INCEPTION", "frequency": "Monthly",
        "bases": DEFAULT_BASES, "needs_universe": False, "time_series": False,
    },
    "cumulative_monthly": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "startdate": "INCEPTION", "frequency": "Monthly",
        "bases": DEFAULT_BASES, "needs_universe": False, "time_series": True,
    },
    "monthly_raw_returns": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "startdate": "INCEPTION", "frequency": "Monthly",
        "bases": DEFAULT_BASES, "needs_universe": False, "time_series": True,
    },
    # --- peer tables: gross only ------------------------------------------
    # Peer universes are conventionally gross, so ranking a net return against a gross
    # universe is not a like-for-like comparison. See the compliance note above — if a peer
    # ranking counts as a performance presentation, change these to DEFAULT_BASES.
    "peer_multi_horizon": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "startdate": "INCEPTION", "frequency": "Monthly",
        "bases": ("gross",), "needs_universe": True, "time_series": False,
    },
    "peer_calendar_year": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "startdate": "INCEPTION", "frequency": "Monthly",
        "bases": ("gross",), "needs_universe": True, "time_series": False,
    },
    # --- risk: gross only, fixed window -----------------------------------
    # Risk statistics are not returns, so the net-alongside-gross requirement does not
    # apply. Beta, tracking error and the like are also near-identical gross vs net (a flat
    # fee shifts the level, not the dispersion), so running both would double the units to
    # restate the same numbers.
    "risk_stats_3y": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "startdate": "-3Y", "frequency": "Monthly",   # fixed comparable window, not inception
        "bases": ("gross",), "needs_universe": False, "time_series": False,
    },
    # Planned variant — SAME component_name as risk_stats_3y, inception-to-date window.
    # Name-based resolution means both keys resolve to the same live id automatically.
    # "risk_stats_itd": {
    #     "component_name": "<same name as risk_stats_3y>", "pinned_componentid": None,
    #     "startdate": "INCEPTION", "frequency": "Monthly",
    #     "bases": ("gross",), "needs_universe": False, "time_series": False,
    # },
}

SPAR_DOCUMENT = "Client:/SPAR/HBCM"    # searched for component names every run

# --- date probe: hardened ids, used only to resolve 0CQ --------------------
# convert_pa_dates_to_absolute_format requires enddate + componentid + account (only
# startdate and calendar are optional). Pin a BASIC WEIGHTS component and pass a SINGLE
# account: the smallest, fastest thing that satisfies the endpoint. These are hardcoded on
# purpose — a pinned id needs no lookup, so the date resolves without depending on the
# component-name resolution in Cell 4, and no PA data is fetched.
DATE_PROBE_COMPONENT = "<TODO pinned basic-weights component id>"
DATE_PROBE_ACCOUNT = "<TODO one PA holdings account, path.ACCT>"
PEER_UNIVERSE_CATEGORY = "Custom"      # for the peer universe discovery cell only

# --- execution -------------------------------------------------------------
# 7 tiles x 4 strategies x 2 bases = 56 units. Submitting all of them as one calculation
# is all-or-nothing and pushes the long-running window; batching per tile gives 8 units
# per calc, isolates failures to one tile, and stays clear of the ~5-10 concurrent-calc
# ceiling. Set False only for a small ad hoc run.
BATCH_PER_TILE = True

# --- OneLake target --------------------------------------------------------
WORKSPACE_ID = "1b9fac18-9d75-4437-ab6c-b6ba44ff46a8"   # HBCM - Production
LAKEHOUSE_ID = "7cdf13b1-4586-4a02-b8ff-72fcf6db1277"   # hbcm_datahub (schema-enabled)
# NOTE the `.Lakehouse` suffix on the item id — required in a OneLake ABFSS path.
# Without it the write does not land in the lakehouse's managed area, so the table
# never registers and never appears in the SQL endpoint or Power BI.
ONELAKE = (f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/"
           f"{LAKEHOUSE_ID}.Lakehouse")
TABLE_PATH = f"{ONELAKE}/Tables/factset/spar_composite_returns"
TABLE_RUN_LOG = f"{ONELAKE}/Tables/factset/factset_run_log"
TABLE_VINTAGE_MANIFEST = f"{ONELAKE}/Tables/factset/vintage_manifest"
TABLE_DIM_VINTAGE = f"{ONELAKE}/Tables/factset/dim_vintage"   # shared with PA
RAW_DIR = f"{ONELAKE}/Files/raw/spar"


# Fail fast on config that cannot possibly work, before burning an API call.
for _c, _s in STRATEGIES.items():
    assert _s["bench_group"] in BENCHMARK_GROUPS, \
        f"{_c}: unknown bench_group {_s['bench_group']!r}"
for _t, _cfg in TILES.items():
    assert _cfg["bases"], f"tile {_t} has no fee bases"
    assert set(_cfg["bases"]) <= {"gross", "net"}, f"tile {_t}: bad bases {_cfg['bases']}"
    if _cfg["needs_universe"]:
        _bad = [c for c in STRATEGIES if not strategy_config(c).get("universe")]
        assert not _bad, f"tile {_t} needs a universe; missing for {_bad}"

N_UNITS = sum(len(c["bases"]) for c in TILES.values()) * len(STRATEGIES)
print(f"{len(TILES)} tiles x {len(STRATEGIES)} strategies x per-tile bases = "
      f"{N_UNITS} units"
      f"{f' in {len(TILES)} batches' if BATCH_PER_TILE else ' in 1 calculation'}")
for _t, _cfg in TILES.items():
    print(f"  {_t:<24} bases={'+'.join(_cfg['bases']):<12} start={_cfg['startdate']}")
print(f"as-of {AS_OF_RELATIVE} resolves in Cell 4b")

In [ ]:
# === Cell 4: resolve component ids BY NAME — every run ====================
# Component ids are not stable. Editing and re-saving a SPAR component in the workstation
# can mint a new id, and a stale hardcoded id fails as a 400 with no hint that the id is
# the problem. So the tile's workstation NAME is the contract, and the id is looked up
# fresh on every run. This is pipeline code, not discovery.

comp_api = components_api.ComponentsApi(api_client)

def _field(obj, name):
    """SDK models allow attribute or dict-style access depending on construction."""
    if hasattr(obj, name):
        return getattr(obj, name)
    try:
        return obj.get(name)
    except AttributeError:
        return None

def resolve_component_ids(document: str = SPAR_DOCUMENT):
    """tile name -> live component id, plus any drift against pinned ids."""
    summary, headers = call_with_headers(comp_api.get_spar_components, document=document)
    RUN_META.setdefault("headers", {}).update(headers)

    by_name = {}
    for cid, meta in (summary.data or {}).items():
        by_name.setdefault(_field(meta, "name"), []).append(cid)

    resolved, drift, missing = {}, [], []
    for tile, cfg in TILES.items():
        name = cfg["component_name"]
        hits = by_name.get(name, [])
        if len(hits) != 1:
            # Ambiguous or absent — refuse to guess which component was meant.
            missing.append((tile, name, len(hits)))
            continue
        cid = hits[0]
        resolved[tile] = cid
        pinned = cfg.get("pinned_componentid")
        if pinned and pinned != cid:
            drift.append((tile, name, pinned, cid))

    return resolved, drift, missing, by_name

RESOLVED_COMPONENTS, COMPONENT_DRIFT, COMPONENT_MISSING, COMPONENTS_BY_NAME = \
    resolve_component_ids()

for tile, cid in RESOLVED_COMPONENTS.items():
    print(f"{tile:<24} {cid}  ({TILES[tile]['component_name']})")

if COMPONENT_DRIFT:
    # Not fatal — a re-saved component is normal. But it must be loud, because it means
    # the component may have been redefined, so the columns could have moved too.
    print("\n*** COMPONENT ID DRIFT — verify the component still returns what you expect,")
    print("*** then update pinned_componentid in Cell 3:")
    for tile, name, was, now in COMPONENT_DRIFT:
        print(f"    {tile}: {name!r}  {was} -> {now}")

if COMPONENT_MISSING:
    print("\nUnresolved tiles (name absent from the document, or not unique):")
    for tile, name, n in COMPONENT_MISSING:
        print(f"    {tile}: {name!r} matched {n} components")
    print("\nComponent names available in", SPAR_DOCUMENT)
    for name, cids in sorted(COMPONENTS_BY_NAME.items(), key=lambda kv: str(kv[0])):
        print(f"    {name!r}: {cids}")

assert not COMPONENT_MISSING, "every tile needs exactly one matching component name"
RUN_META["components"] = dict(RESOLVED_COMPONENTS)
RUN_META["component_drift"] = [list(d) for d in COMPONENT_DRIFT]
RUN_META["component_names"] = {t: c["component_name"] for t, c in TILES.items()}

In [ ]:
# === Cell 4b: resolve 0CQ to an absolute date — PA is the authority =======
# SPAR has no DatesApi, so PA resolves the token and SPAR sends the absolute result. Three
# sources, in order of preference:
#   1. the date the PA notebook already published to OneLake (guarantees both agree)
#   2. a direct PA DatesApi call (needs PA_PROBE_COMPONENT + PA_PROBE_ACCOUNT)
#   3. a locally computed quarter end (last resort; recorded as such)

def _fallback_quarter_end(today=None):
    """Most recent COMPLETED calendar quarter end."""
    d = today or dt.date.today()
    qe = dt.date(d.year, ((d.month - 1) // 3) * 3 + 1, 1) - dt.timedelta(days=1)
    return qe.strftime("%Y%m%d")

def _asof_from_onelake(relative):
    try:
        raw = notebookutils.fs.head(f"{ONELAKE}/Files/raw/_asof/{relative}.json", 4096)
        rec = json.loads(raw)
        if rec.get("absolute"):
            return str(rec["absolute"]), f"PA notebook via OneLake ({rec.get('resolved_utc')})"
    except Exception:
        pass
    return None, None

def _asof_from_pa(relative):
    """Pinned basic-weights component + a single account: the cheapest call that satisfies
    the endpoint's required args, so this stays fast and needs no component lookup."""
    if "TODO" in DATE_PROBE_COMPONENT or "TODO" in DATE_PROBE_ACCOUNT:
        return None, None
    try:
        resp, headers = call_with_headers(
            pa_d_api.convert_pa_dates_to_absolute_format,
            enddate=relative, componentid=DATE_PROBE_COMPONENT,
            account=DATE_PROBE_ACCOUNT,
        )
        RUN_META.setdefault("headers", {}).update(headers)
        end = _field(resp.data, "enddate")
        if end:
            return str(end), "PA DatesApi (direct)"
    except Exception as e:
        print(f"PA date conversion failed: {e!r}")
    return None, None

AS_OF_ABS, AS_OF_SOURCE = _asof_from_onelake(AS_OF_RELATIVE)
if AS_OF_ABS is None:
    AS_OF_ABS, AS_OF_SOURCE = _asof_from_pa(AS_OF_RELATIVE)
if AS_OF_ABS is None:
    AS_OF_ABS = _fallback_quarter_end()
    AS_OF_SOURCE = "local fallback (PA unavailable — set DATE_PROBE_* or run the PA notebook)"

# SENT vs LABELLED are deliberately different. The request stays dynamic so the scheduled
# job needs no maintenance; the label is the absolute date PA resolved, so the data carries
# a real date and can be validated against what came back (Cell 8).
END_DATE = AS_OF_ABS if SEND_ABSOLUTE_END_DATE else AS_OF_RELATIVE
asof_tag = AS_OF_ABS

_local = _fallback_quarter_end()
print(f"{AS_OF_RELATIVE} -> {AS_OF_ABS}   source: {AS_OF_SOURCE}")
print(f"enddate SENT: {END_DATE!r}   asof_date LABELLED: {asof_tag}")
if AS_OF_ABS != _local:
    # Not necessarily wrong — PA may use a trading-calendar quarter end. Worth knowing.
    print(f"NOTE: differs from the locally computed quarter end ({_local}). PA's answer wins.")

RUN_META.update({
    "asof_relative": AS_OF_RELATIVE,
    "asof_absolute": AS_OF_ABS,
    "asof_source": AS_OF_SOURCE,
    "asof_local_computed": _local,
    "currency": CURRENCY,
    "end_date_sent": END_DATE,
    "send_absolute_end_date": SEND_ABSOLUTE_END_DATE,
})

In [ ]:
# === Cell 4c: DISCOVERY — run interactively once, then leave it alone ======
# Resolves the TODOs in Cell 3. Not part of the scheduled path.

acct_api = accounts_api.AccountsApi(api_client)
bmk_api = benchmarks_api.BenchmarksApi(api_client)   # comp_api is created in Cell 4
peer_api = spar_peer_universe_api.SPARPeerUniverseApi(api_client)
freq_api = frequencies_api.FrequenciesApi(api_client)

# (a) Return types per ACCT -> STRATEGIES[...]["returntype"]
#     This is the authoritative answer on gross vs net. ReturnType carries (name, id);
#     the `id` is what goes in SPARIdentifier.returntype and is often a numeric code,
#     not the literal string "Net".
for code, s in STRATEGIES.items():
    path = s["acct"] if s["acct"].endswith(".ACCT") else f"{s['acct']}.ACCT"
    try:
        resp = acct_api.get_spar_returns_type(path)   # URL-encoded account path
        types = [(rt.get("name"), rt.get("id")) for rt in (resp.data.returns_type or [])]
        print(f"OK   {code:8s} {path:36s} -> {types}")
    except fds.sdk.SPAREngine.ApiException as e:
        print(f"FAIL {code:8s} {path:36s} -> {e.status} {e.body}")

# (b) Component names -> TILES[...]["component_name"]. Cell 4 already resolves these on
#     every run and prints the full name list if any tile fails to match, so there is
#     nothing to do here. Kept as a pointer.

# (c) Peer universes -> STRATEGIES[...]["universe"] (needed by the two peer tiles).
#     `category` is required; `name` and `directory` are optional filters.
try:
    print(peer_api.get_list_of_peer_universe(PEER_UNIVERSE_CATEGORY))
except fds.sdk.SPAREngine.ApiException as e:
    print(f"peer universe lookup failed: {e.status} {e.body}")

# (d) Valid frequency ids -> TILES[...]["frequency"]. Calendar-year and multi-horizon
#     tiles may want something other than Monthly.
try:
    print(freq_api.get_spar_frequencies())
except fds.sdk.SPAREngine.ApiException as e:
    print(f"frequencies lookup failed: {e.status} {e.body}")

# (e) Confirm a benchmark id resolves and see its valid prefixes -> ["benchmark"]
#   print(bmk_api.get_spar_benchmark_by_id(id="<benchmark id>"))

# (f) READ THE SAVED CONFIG OFF EACH COMPONENT.
#     SPARComponent exposes the accounts and benchmark SAVED IN THE DOCUMENT, each as a
#     SPARComponentIdentifier carrying id + returntype + prefix. That is exactly the three
#     fields STRATEGIES and BENCHMARK_GROUPS need, so the document path plus component
#     names is enough to fill in accts, prefixes, return types and benchmark ids without
#     hand-transcribing them from the workstation.
#
#     Run once, map the output into Cell 3, then skip. Note the returntype shown is
#     whatever the component was saved with — for a gross/net pair you still need BOTH
#     values, which come from (a) above.
for _tile, _cid in RESOLVED_COMPONENTS.items():
    try:
        _c = comp_api.get_spar_component_by_id(id=_cid).data
    except fds.sdk.SPAREngine.ApiException as e:
        print(f"{_tile}: component lookup failed {e.status} {e.body}")
        continue
    print(f"\n=== {_tile} ({_field(_c, 'name')}) ===")
    print(f"  path      {_field(_c, 'path')}")
    print(f"  currency  {_field(_c, 'currency_iso_code')}")
    for _a in (_field(_c, "accounts") or []):
        print(f"  ACCOUNT   id={_field(_a, 'id')!r} "
              f"returntype={_field(_a, 'returntype')!r} prefix={_field(_a, 'prefix')!r}")
    _b = _field(_c, "benchmarks")
    for _bi in (_b if isinstance(_b, list) else [_b] if _b else []):
        print(f"  BENCHMARK id={_field(_bi, 'id')!r} "
              f"returntype={_field(_bi, 'returntype')!r} prefix={_field(_bi, 'prefix')!r}")

In [ ]:
# === Cell 5: build the calculation units ==================================
# One unit per (tile, strategy, basis) = 7 x 4 x 2 = 56.
#
# LC and LCS share a benchmark and peer universe, so it is tempting to put both in one
# unit's `accounts` list. Don't: SPARCalculationParameters carries ONE `dates` object per
# unit, so accounts sharing a unit share a start date — which defeats resolving INCEPTION
# per strategy. (SPAR does expose `useeachportfolioinception` for exactly that case, but
# it hands the window to the engine, so the request no longer states what it computed and
# there is nothing to verify against. Explicit beats implicit here.) The shared benchmark
# is deduplicated in config instead, via BENCHMARK_GROUPS.

def account_identifier(code: str, basis: str) -> SPARIdentifier:
    s = strategy_config(code)
    if SYMBOL_SUFFIX_MODE:
        # Separate gross/net symbols (QR / $$Performance.ofdb layout).
        return SPARIdentifier(id=f"{s['acct']}{SUFFIX[basis]}", returntype=None,
                              prefix=s["prefix"])
    # One ACCT carrying both streams — pick the stream by return type.
    return SPARIdentifier(id=s["acct"], returntype=s["returntype"][basis],
                          prefix=s["prefix"])

def resolve_startdate(tile_cfg: dict) -> tuple:
    """(startdate, useeachportfolioinception) for a tile.

    "INCEPTION" hands the window to the engine: each strategy starts at its own earliest
    available monthly data. SPAR still requires a startdate field even then, so a wide
    floor goes in and the flag supersedes it per portfolio. The ACTUAL dates are read back
    off the response in Cell 8 rather than assumed here.
    """
    sd = tile_cfg["startdate"]
    if sd == "INCEPTION":
        return INCEPTION_FALLBACK_START, True
    return sd, False

def build_unit(tile_name: str, tile_cfg: dict, code: str, basis: str) -> SPARCalculationParameters:
    s = strategy_config(code)
    bmk = s["benchmark"]
    startdate, use_inception = resolve_startdate(tile_cfg)
    kwargs = dict(
        componentid=RESOLVED_COMPONENTS[tile_name],   # resolved by name in Cell 4
        accounts=[account_identifier(code, basis)],
        # SPAR benchmarks are the OFFICIAL index return streams. Returns-based analysis
        # needs no constituents, so there is no entitlement obstacle here — unlike PA, which
        # uses iShares ETF proxies because it needs holdings.
        benchmark=SPARIdentifier(id=bmk["id"], returntype=bmk["returntype"],
                                 prefix=bmk["prefix"]),
        dates=SPARDateParameters(
            startdate=startdate,
            enddate=END_DATE,
            frequency=tile_cfg["frequency"],
            useeachportfolioinception=use_inception,
        ),
        currencyisocode=CURRENCY,
    )
    # Send universeid only for the peer tiles. Passing None on a non-peer tile would
    # override the component's saved universe with nothing.
    if tile_cfg["needs_universe"]:
        kwargs["universeid"] = s["universe"]
    return SPARCalculationParameters(**kwargs)

def unit_key(tile_name: str, code: str, basis: str) -> str:
    return f"{tile_name}__{code}__{basis}"

def make_root(units: dict) -> SPARCalculationParametersRoot:
    return SPARCalculationParametersRoot(
        data=units,
        meta=CalculationMeta(
            contentorganization="SimplifiedRow",
            stach_content_organization="SimplifiedRow",
            contenttype="Json",
            format="JsonStach",
        ),
    )

UNIT_KEYS = {}          # unit_key -> (tile_name, strategy_code, basis)
batches = []            # list[(batch_label, SPARCalculationParametersRoot)]
all_units = {}

for tile_name, tile_cfg in TILES.items():
    tile_units = {}
    for code in STRATEGIES:
        for basis in tile_cfg["bases"]:        # per tile, not global
            key = unit_key(tile_name, code, basis)
            tile_units[key] = build_unit(tile_name, tile_cfg, code, basis)
            UNIT_KEYS[key] = (tile_name, code, basis)
    all_units.update(tile_units)
    if BATCH_PER_TILE:
        batches.append((tile_name, make_root(tile_units)))

if not BATCH_PER_TILE:
    batches = [("all", make_root(all_units))]

# Show the resolved window per tile x strategy. Read down a column: the inception tiles
# must show that strategy's own date, and only risk_stats should show -3Y.
print(f"{len(all_units)} units in {len(batches)} batch(es)")
print(f"enddate sent: {END_DATE!r}\n")
print(f"{'tile':<24}{'bases':<14}{'startdate sent':<18}each-port-inception")
for tile_name, tile_cfg in TILES.items():
    sd, uei = resolve_startdate(tile_cfg)
    print(f"{tile_name:<24}{'+'.join(tile_cfg['bases']):<14}{sd:<18}{uei}")
print("\nINCEPTION tiles: the engine picks each strategy's own start from its available")
print("monthly data. Cell 8 reports the dates that actually came back.\n")
for code in STRATEGIES:
    s = strategy_config(code)
    print(f"{code:<6} bench={s['benchmark']['prefix']}{s['benchmark']['id']:<24} "
          f"universe={s['universe']:<20} group={s['bench_group']}")

In [ ]:
# === Cell 6: submit + poll ================================================
# Multi-unit calculations ALWAYS return 202 regardless of the deadline header, so the
# polling loop is the normal path here, not the exception.

def run_spar(params_root, deadline=10, poll_interval=3, timeout=900):
    """Returns (calc_id, list[(unit_id, result_or_None, status)], per-unit metadata)."""
    wrapper, headers = call_with_headers(
        calc_api.post_and_calculate,
        x_fact_set_api_long_running_deadline=deadline,
        spar_calculation_parameters_root=params_root,
    )
    RUN_META.setdefault("headers", {}).update(headers)
    code = wrapper.get_status_code()
    if code == 200:
        status_root = wrapper.get_response_200()
    elif code == 201:
        status_root = wrapper.get_response_201()
    elif code == 202:
        status_root = wrapper.get_response_202()
        calc_id = status_root.data.calculationid
        deadline_at = time.time() + timeout
        while True:
            if time.time() > deadline_at:
                calc_api.cancel_calculation_by_id(id=calc_id)
                raise TimeoutError(f"calc {calc_id} exceeded {timeout}s (cancelled)")
            poll = calc_api.get_calculation_status_by_id(id=calc_id)
            if poll.get_status_code() == 200:
                status_root = poll.get_response_200()
                break
            if poll.get_status_code() != 202:
                raise RuntimeError(f"unexpected poll status {poll.get_status_code()}")
            time.sleep(poll_interval)
    else:
        raise RuntimeError(f"unexpected submit status {code}")

    calc_id = status_root.data.calculationid
    out, unit_meta = [], {}
    for unit_id, unit_status in (status_root.data.units or {}).items():
        st = _field(unit_status, "status")
        # Whatever the engine reports per unit is worth keeping — it is the only record of
        # why a unit came back empty.
        unit_meta[unit_id] = {
            "status": st,
            "error": str(_field(unit_status, "error") or ""),
            "progress": str(_field(unit_status, "progress") or ""),
        }
        if st != "Success":
            out.append((unit_id, None, st))
            continue
        res, rheaders = call_with_headers(
            calc_api.get_calculation_unit_result_by_id, id=calc_id, unit_id=unit_id)
        RUN_META.setdefault("headers", {}).update(rheaders)
        out.append((unit_id, res, st))
    return calc_id, out, unit_meta

results, calc_ids, batch_errors, UNIT_META = [], {}, {}, {}
for label, root in batches:
    try:
        cid, batch_results, unit_meta = run_spar(root)
        calc_ids[label] = cid
        UNIT_META.update(unit_meta)
        results.extend(batch_results)
        n_bad = sum(1 for _, r, _ in batch_results if r is None)
        print(f"{label:24s} calc={cid} ok={len(batch_results) - n_bad} failed={n_bad}")
    except Exception as e:
        # One bad tile should not cost the other six. Record and continue.
        batch_errors[label] = repr(e)
        print(f"{label:24s} BATCH FAILED: {e!r}")

failed = [(u, s) for u, r, s in results if r is None]
for u, s in failed:
    print(f"  FAILED unit {u}: {s} — {UNIT_META.get(u, {}).get('error', '')}")

RUN_META["calculation_ids"] = dict(calc_ids)
RUN_META["unit_status"] = UNIT_META
RUN_META["batch_errors"] = dict(batch_errors)
print("\ncaptured headers:")
for k, v in RUN_META.get("headers", {}).items():
    print(f"    {k}: {v}")

# Log calc ids alongside X-DataDirect-Request-Key for any FactSet support ticket —
# use the *_with_http_info variants when you need the response headers.
print(f"\n{len(results) - len(failed)} usable units; "
      f"{len(failed)} failed units; {len(batch_errors)} failed batches")
assert results and not failed and not batch_errors, (
    "resolve failures before writing to the lakehouse — a partial snapshot is worse "
    "than none, because it looks complete downstream"
)

In [ ]:
# === Cell 7: raw landing (write-once audit copy) ==========================
# FactSet earns a raw layer: calls are slow and async, STACH reshaping is fiddly, and
# vendor restatements mean the same call replayed later does NOT return what it
# originally returned. That matters for GIPS and Marketing Rule substantiation.
# Files, not Delta — raw stays invisible to the SQL endpoint, which is correct.

# asof_tag was set in Cell 4b from PA's resolved date — never a relative token.
for unit_id, res, _ in results:
    notebookutils.fs.put(
        f"{RAW_DIR}/asof={asof_tag}/{unit_id}.json",
        json.dumps(res.to_dict(), default=str),
        True,
    )
print(f"landed {len(results)} raw payloads under {RAW_DIR}/asof={asof_tag}/")

In [ ]:
# === Cell 8: STACH -> tidy DataFrame ======================================
# Parsing and typing both come from stach_parse / type_from_schema in Cell 2, which read
# the package's own column definitions instead of guessing from names or values.
frames, collisions, schemas, undeclared = [], set(), {}, set()
for unit_id, res, _ in results:
    tile_name, code, basis = UNIT_KEYS[unit_id]
    s, tile_cfg = strategy_config(code), TILES[tile_name]
    for i, (tid, df, schema, levels) in enumerate(stach_parse(res)):
        df, tp = type_from_schema(df, schema)
        undeclared.update(tp["undeclared"])
        schemas[f"{unit_id}[{i}]"] = {"table_id": tid, **{k: v for k, v in tp.items() if v}}
        # Provenance first, so the grain is legible without joining anything.
        # start_date is the resolved value actually sent, not the "INCEPTION" token.
        provenance = [
            ("asof_date",     asof_tag),
            ("tile",          tile_name),
            ("strategy_code", code),
            ("strategy",      s["label"]),
            ("fee_basis",     basis),
            ("account_id",    s["acct"]),
            ("benchmark_id",  s["benchmark"]["id"]),
            ("bench_group",   s["bench_group"]),
            ("universe_id",   s["universe"] if tile_cfg["needs_universe"] else None),
            ("start_date",    resolve_startdate(tile_cfg, code)),
            ("frequency",     tile_cfg["frequency"]),
            ("end_date",      END_DATE),
            ("asof_relative", AS_OF_RELATIVE),
            ("componentid",   RESOLVED_COMPONENTS[tile_name]),
            ("component_name", tile_cfg["component_name"]),
            ("currency",      CURRENCY),
            ("calculation_id", calc_ids.get(tile_name if BATCH_PER_TILE else "all", "")),
            ("table_ix",      i),
            ("stach_table_id", tid),
        ]
        df, clashed = dedupe_columns(df, {c for c, _ in provenance})
        if clashed:
            collisions.update(clashed)
        for pos, (col, val) in enumerate(provenance):
            df.insert(pos, col, val)
        frames.append(df)

tidy = pd.concat(frames, ignore_index=True)
if collisions:
    # Not fatal, but you want to know a STACH column was renamed out of the way.
    print(f"STACH columns renamed to avoid clashing with provenance: {sorted(collisions)}"
          f" (suffixed _src)")

print(tidy.shape)
print(tidy.groupby(["tile", "fee_basis"], dropna=False).size())
# Confirms the inception reset landed: one distinct start_date per strategy on the
# inception tiles, and a single -3Y-derived window on the risk tile.
print("\nstart_date by strategy:")
print(tidy.groupby(["strategy_code", "tile"])["start_date"].unique())

# --- derive the REAL window from each time-series response ----------------
# The engine chose each strategy's start from its own available monthly data, so the request
# does not state what was actually computed. The response does. For every
# (time-series tile x strategy) take min(date), max(date) and the row count — that IS the
# confirmed inception, the confirmed end, and the as-of.
#
# The four series are deliberately ragged: LC, SMID, LCS and CONC each begin on a different
# date with different row counts. That is correct and is not padded over.
# The date column is whatever STACH DECLARED as a date — no name matching, so a component
# that calls it "Period End" instead of "Date" still works.
_declared_dates = sorted({c for sch in schemas.values() for c in sch.get("date", [])})
_ts_tiles = [t for t, c in TILES.items() if c.get("time_series")]
_ts = tidy[tidy["tile"].isin(_ts_tiles)] if _ts_tiles else tidy.iloc[0:0]
_date_col = next((c for c in _declared_dates if c in _ts.columns), None)
print(f"columns STACH declared as dates: {_declared_dates or '(none)'}")
if undeclared:
    print(f"columns with no declared STACH type (fell back to sniffing): {sorted(undeclared)}")

SERIES_WINDOWS = None
if _date_col is None:
    # Column naming comes from the component, so it cannot be known in advance.
    print("\n*** the time-series tiles declare no date column, so the confirmed window is")
    print("*** unknown. Check the component's column definitions.")
    print("    columns:", list(_ts.columns))
    if len(_ts):
        print(_ts.groupby(["tile", "strategy_code"]).size())
else:
    _d = pd.to_datetime(_ts[_date_col], errors="coerce")
    SERIES_WINDOWS = (
        _ts.assign(_d=_d)
           .groupby(["tile", "strategy_code", "fee_basis"], dropna=False)["_d"]
           .agg(first_period="min", last_period="max", periods="count")
           .reset_index()
    )
    print(f"\nconfirmed window per time-series tile x strategy (date col: {_date_col!r}):")
    print(SERIES_WINDOWS.to_string(index=False))

    # The as-of is the maximum date across every series. Compare it to the date PA resolved
    # for 0CQ — a mismatch means the label on every row is wrong, which is the one error
    # here that would propagate silently into a tearsheet.
    _max = SERIES_WINDOWS["last_period"].max()
    _max_tag = _max.strftime("%Y%m%d") if pd.notna(_max) else None
    print(f"\nmax date in data: {_max_tag}   asof_date label: {asof_tag}   "
          f"({AS_OF_SOURCE})")
    if _max_tag and _max_tag != asof_tag:
        print("*** MISMATCH: the data ends on a different date than the label claims.")
        print("*** Either 0CQ resolved to a quarter the data does not reach (a composite is")
        print("*** not yet updated), or the label is wrong. Resolve before relying on this.")

    # Per-strategy inception, confirmed from data rather than configured anywhere.
    _inc = (SERIES_WINDOWS.groupby("strategy_code")["first_period"].min()
                          .dt.strftime("%Y-%m-%d"))
    print("\nconfirmed inception per strategy (earliest monthly data):")
    print(_inc.to_string())
    RUN_META["confirmed_inception"] = _inc.to_dict()
    RUN_META["confirmed_max_date"] = _max_tag
    RUN_META["date_column"] = _date_col

    # Gross and net of the same strategy must span the same periods; if they don't, one
    # stream is incomplete and any gross-vs-net comparison is off by the missing months.
    _pivot = SERIES_WINDOWS.pivot_table(index=["tile", "strategy_code"],
                                        columns="fee_basis", values="periods")
    if {"gross", "net"} <= set(_pivot.columns):
        _uneven = _pivot[_pivot["gross"] != _pivot["net"]]
        if len(_uneven):
            print("\n*** gross and net period counts differ — one stream is incomplete:")
            print(_uneven.to_string())

    # --- cross-tile agreement -------------------------------------------
    # cumulative_monthly and monthly_raw_returns are two views of the SAME monthly stream
    # for the same strategy, so their first period, last period and count must match. A
    # disagreement means one tile's component has its own saved date range overriding the
    # request, or one call silently returned a truncated series — either way the two would
    # tell different stories about the same composite.
    if len(_ts_tiles) > 1:
        _x = (SERIES_WINDOWS.pivot_table(
                  index=["strategy_code", "fee_basis"], columns="tile",
                  values=["first_period", "last_period", "periods"], aggfunc="first"))
        print("\ncross-tile agreement per strategy (the two monthly tiles must match):")
        print(_x.to_string())
        _bad = []
        for field in ("first_period", "last_period", "periods"):
            sub = _x[field]
            if sub.shape[1] > 1:
                differs = sub.nunique(axis=1, dropna=False) > 1
                for idx in sub.index[differs]:
                    _bad.append((field, idx, sub.loc[idx].to_dict()))
        if _bad:
            print("\n*** TIME-SERIES TILES DISAGREE:")
            for field, idx, vals in _bad:
                print(f"    {field} {idx}: {vals}")
            print("*** Same strategy, same monthly data — these must agree. Check whether a")
            print("*** component's saved date range is overriding the request.")
        else:
            print("  OK  both monthly tiles agree on start, end and period count")
        RUN_META["cross_tile_date_mismatches"] = [
            [f, list(i) if isinstance(i, tuple) else i, {str(k): str(v) for k, v in d.items()}]
            for f, i, d in _bad
        ]

    # --- horizon / calendar-year misalignment ----------------------------
    # The ragged inceptions make some rows meaningless rather than merely empty: a "5 Year"
    # horizon for a composite with three years of history, or a calendar-year return for the
    # partial year it launched in. Both look like real numbers in a visual.
    _months = ((pd.Timestamp(asof_tag).to_period("M") -
                SERIES_WINDOWS.groupby("strategy_code")["first_period"]
                              .min().dt.to_period("M")).apply(lambda x: x.n))
    print("\nmonths of history per strategy (governs which horizons are meaningful):")
    print(_months.to_string())
    RUN_META["months_of_history"] = {k: int(v) for k, v in _months.items()}

    _label_col = next((c for c in ("label", "period", "horizon", "name", "description",
                                   "row_label") if c in tidy.columns), None)
    if _label_col:
        import re as _re
        def _horizon_months(lbl):
            m = _re.search(r"(\d+)\s*(y|yr|year|m|mo|month)", str(lbl), _re.I)
            if not m:
                return None
            n_ = int(m.group(1))
            return n_ * 12 if m.group(2).lower().startswith("y") else n_
        _mh = tidy[tidy["tile"].isin(["multi_horizon_returns", "peer_multi_horizon"])]
        rows = []
        for _, r in _mh.iterrows():
            need = _horizon_months(r[_label_col])
            have = _months.get(r["strategy_code"])
            if need and have is not None and need > have:
                rows.append((r["strategy_code"], r["tile"], r[_label_col], need, int(have)))
        if rows:
            uniq = sorted(set(rows))
            print(f"\n*** {len(uniq)} horizon rows exceed the strategy's history "
                  f"(label col {_label_col!r}) — exclude or annotate these, do not show them")
            for c_, t_, l_, n_, h_ in uniq[:20]:
                print(f"    {c_:<6} {t_:<22} {str(l_):<16} needs {n_}m, has {h_}m")
            RUN_META["horizons_exceeding_history"] = [list(u) for u in uniq]
        else:
            print("  OK  no horizon label exceeds its strategy's available history")

        # Partial first calendar year: inception mid-year means that year's "annual" return
        # covers only part of the year.
        _first_year = (SERIES_WINDOWS.groupby("strategy_code")["first_period"].min())
        partial = {c: d for c, d in _first_year.items() if pd.notna(d) and d.month > 1}
        if partial:
            print("\nNOTE partial first calendar year (inception after January) — the "
                  "calendar-year row for that year is a stub, not a full-year return:")
            for c_, d_ in partial.items():
                print(f"    {c_:<6} first period {d_.date()} -> {d_.year} is partial")
            RUN_META["partial_first_year"] = {c: int(d.year) for c, d in partial.items()}
    else:
        print("\nNOTE no period-label column found, so horizon coverage could not be "
              "checked. Add the real name to the candidate list if you want that check.")

# --- type the output ------------------------------------------------------
# Applied last, so the derived date column is known and can be typed as a real date.
# asof_date stays a YYYYMMDD string because the Delta delete predicate keys on it;
# asof_date_iso is the proper date for a Power BI relationship.
# Typing was applied per table from its own schema; only the vintage key is added here.
tidy["asof_date_iso"] = pd.to_datetime(asof_tag, format="%Y%m%d").date()
_num = [c for c in tidy.columns if str(tidy[c].dtype) in ("Float64", "Int64", "float64")]
print(f"\ntyped columns: {len(_num)} numeric, {len(_declared_dates)} declared date")
if not _num:
    print("*** every column landed as text — Power BI would need a type conversion per")
    print("*** measure. Check the component's declared column types.")
RUN_META["stach_schemas"] = schemas
RUN_META["undeclared_types"] = sorted(undeclared)

display(tidy.head(20))

In [ ]:
# === Cell 9: idempotent write to hbcm_datahub =============================
# delete-then-append on asof_date, so a pipeline retry or manual re-run replaces the
# snapshot instead of doubling it. mode="append" alone would silently duplicate.
# --- validate the join key before anything is written --------------------
print("validating factset.spar_composite_returns")
_codes = assert_strategy_key(tidy, "spar_composite_returns")
print(f"  OK   strategy_code present and canonical: {_codes}")
# Grain is per-tile: each component has its own row axis, so uniqueness is checked within a
# tile rather than across the table.
for _t in sorted(tidy["tile"].unique()):
    _sub = tidy[tidy["tile"] == _t]
    _axis = next((c for c in ("label", "period", "horizon", "name", "row_label")
                  if c in _sub.columns), None)
    _keys = ["asof_date", "tile", "strategy_code", "fee_basis", "table_ix"] + \
            ([_axis] if _axis else [])
    assert_unique_grain(_sub, _keys, f"spar_composite_returns[{_t}]")
RUN_META["validated_codes"] = _codes

if table_exists(TABLE_PATH):
    DeltaTable(TABLE_PATH).delete(f"asof_date = '{asof_tag}'")
    mode = "append"
else:
    mode = "overwrite"          # first write creates the table

write_deltalake(TABLE_PATH, tidy, mode=mode, schema_mode="merge")
print(f"wrote {len(tidy)} rows to factset.spar_composite_returns (mode={mode})")
print("\ndtypes landed (numeric measures should NOT be string):")
print(tidy.dtypes.value_counts().to_string())

# No partitioning: this table is kilobytes. Partitioning a small table costs more in
# metadata than it saves in pruning.
# Re-frame any Direct Lake semantic model BEFORE running VACUUM — vacuuming files a
# framed model still points at gives users query errors on missing files.
# Order is always: write -> frame -> vacuum.

In [ ]:
# === Cell 10: land the run log ============================================
# One row per run, in the same table the PA notebook writes to. Request keys are the
# difference between "we'll investigate" and "we know exactly what you sent" on a FactSet
# support ticket, and they exist only at call time.

RUN_META["run_finished_utc"] = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")
RUN_META["rows_written"] = len(tidy)
RUN_META["units_ok"] = len(results) - len(failed)
RUN_META["units_failed"] = len(failed)
RUN_META["tiles"] = list(TILES)
RUN_META["strategies"] = list(STRATEGIES)

flat = {"asof_date": asof_tag}
for k, v in RUN_META.items():
    flat[k] = v if isinstance(v, (str, int, float, bool)) or v is None \
        else json.dumps(v, default=str)

run_log = pd.DataFrame([flat]).astype("string")
mode = "append" if table_exists(TABLE_RUN_LOG) else "overwrite"   # append-only history
write_deltalake(TABLE_RUN_LOG, run_log, mode=mode, schema_mode="merge")
print(f"logged run to factset.factset_run_log (mode={mode})")
for k in ("asof_absolute", "asof_source", "units_ok", "units_failed", "rows_written"):
    print(f"    {k}: {RUN_META.get(k)}")

# Bundle this quarter into one auditable vintage and recompute which vintage is latest
# AND complete — that is the one Power BI should read.
record_vintage({"spar_composite_returns": len(tidy)}, "spar_composite_returns")

## Validation checklist

- [ ] Cell 2's version assert passed, so the Environment really is on SDK >= 3.0.0.
- [ ] 56 units expected (7 x 4 x 2); all 7 batches succeeded and no unit failed.
- [ ] Cell 3's two printed dates agree — `0CQ` should resolve to the same quarter end as
      `AS_OF_ABS`. If the data's last period isn't that quarter, the `asof_date` label is
      lying and you want `USE_ABSOLUTE_AS_OF = True`.
- [ ] Gross exceeds net for every strategy and period. If gross == net, the two
      `returntype` values in Cell 3 aren't resolving to distinct streams — re-read Cell 4a.
- [ ] The two peer tiles carry a non-null `universe_id`; the other five carry null.
- [ ] `start_date` on inception tiles matches each composite's actual inception, and
      `-3Y` appears only on the risk tile.
- [ ] Returns tie to the composite performance report — that report, not SPAR, is the
      GIPS authority.

## Known sharp edges

| Symptom | Cause |
|---|---|
| 400, no detail | Wrong `prefix` on account or benchmark. The single most common failure. |
| 400 on dates | `0CQ` or the tile's frequency unsupported for that component — check Cell 4d, or set `USE_ABSOLUTE_AS_OF = True` to sidestep relative-token parsing entirely. |
| Data ends a quarter early or late | `0CQ` resolved to a different quarter than `AS_OF_ABS` assumed. Send the absolute date. |
| 400 mentioning `universeid` | Environment is on an SDK older than 2.1 — the field didn't exist. Cell 2's assert should catch this first. |
| `ImportError` on `spar_peer_universe_api` | Same cause: stale SDK. |
| Gross and net identical | Both `returntype` values resolving to the same stream. |
| Peer tile returns no percentiles | `universe` id wrong, or the component's saved universe conflicts with the override. |
| 404 fetching a result | Calculation id expired (TTL is hours). Resubmit that batch. |
| 429 | Concurrency ceiling, typically 5–10 concurrent calcs per user. Batching per tile is already the mitigation; don't set `BATCH_PER_TILE = False` on the full grid. |
| One tile empty, others fine | That component's saved date range or grouping conflicts with the override. Per-tile batching means it can't take the rest down. |
| Works interactively, fails in pipeline | `%pip` install — bind a Fabric Environment instead. |

## Consuming this from Power BI

The point of the typing and grain work above is that Power Query should need almost nothing.

**Table:** `factset.spar_composite_returns`

**Grain:** one row per `asof_date` x `tile` x `strategy_code` x `fee_basis` x `table_ix` x
whatever the component's own row axis is (period label, peer, statistic). There is no single
natural key across tiles because each tile has a different row axis — filter to one `tile`
first and the grain becomes obvious. Treat `tile` as the table selector, not a slicer.

**Expected M:** `Lakehouse` connector → pick the table → nothing else. Measures are already
`Float64`, `asof_date_iso` is already a date. If you find yourself adding `Table.TransformColumnTypes`
for a measure, that column failed to coerce — fix it here rather than in M, or every report
repeats the fix.

**Joining:** relate `asof_date_iso` to a date dimension, and `strategy_code` to a strategy
dimension. Don't relate on `asof_date` (the YYYYMMDD string) — it exists for the Delta delete
predicate, not for modelling.

**The ragged-series trap:** the monthly tiles start at each strategy's own inception, so a
shared date dimension will show periods where only some composites existed. Guard
cross-strategy aggregates against each strategy's first period. `factset.factset_run_log`
carries the confirmed inception per strategy for exactly this.

## Quarterly refresh, in order

1. **PA notebook first.** It resolves `0CQ` and publishes the absolute date, so both
   pipelines agree on the as-of. SPAR falls back to resolving it itself, but running PA first
   makes agreement structural rather than coincidental.
2. **SPAR notebook.**
3. **Re-frame the Direct Lake semantic model.** A framed model points at a specific Delta
   commit; until it is re-framed it serves last quarter's numbers.
4. **Only then** run any `VACUUM`. Vacuuming files a framed model still references gives
   users query errors on missing files. Order is always write → frame → vacuum.

Both notebooks are re-runnable: the write deletes the current `asof_date` before appending,
so a repeat run replaces the quarter rather than doubling it. Prior quarters are untouched,
and `table_exists()` refuses to guess rather than risking an overwrite of history.

## Sources

Verified against **upstream `FactSet/enterprise-sdk` `main`** (SPAREngine v3, SDK 3.0.0)
— `SPARCalculationsApi.md`, `SPARCalculationParameters.md`, `SPARIdentifier.md`,
`SPARDateParameters.md`, `CalculationMeta.md`, `AccountsApi.md`
(`get_spar_returns_type`), `SPARPeerUniverseApi.md`, `FrequenciesApi.md`,
`ReturnType.md`, plus `BREAKING.md` for the 2026-05-20 Python-SDK bump.

Not against `code/python/SPAREngine/v3/` in this repo, which is pinned at 2.0.3
(2025-07-21) and lacks `universeid` and `SPARPeerUniverseApi`.

Microsoft Learn — [notebook limitations](https://learn.microsoft.com/fabric/data-engineering/notebook-limitation),
[%run](https://learn.microsoft.com/fabric/data-engineering/author-execute-notebook#run-notebooks),
[Python kernel lifecycle](https://learn.microsoft.com/fabric/data-engineering/python-notebook-runtime-lifecycle),
[pandas to lakehouse](https://learn.microsoft.com/fabric/data-engineering/lakehouse-notebook-load-data#load-data-with-pandas-api).